# Summary_Day5.ipynb  
## 경사하강법(GD) 구현 · PyTorch 모델 구조

이번 5차시는 PyTorch로 모델을 만드는 전체 구조를 정리합니다.

핵심 목표:

1. 신경망 개념도와 PyTorch 프로그램 모델의 차이 이해
2. Layer Function, Parameter, Model, Learning 용어 정리
3. `nn.Linear`, `nn.ReLU`, `nn.Sequential` 사용법 이해
4. 입력 Tensor가 Layer를 지나며 shape이 어떻게 바뀌는지 확인
5. PyTorch 학습의 3대 요소 이해: `net`, `criterion`, `optimizer`
6. 학습 4단계 순환 이해: 예측 → 손실 → 경사 → 수정
7. 활성화 함수가 왜 필요한지 실험으로 확인
8. 선형 모델, 활성화 없는 깊은 모델, ReLU가 있는 모델 비교
9. 기울기 소실/폭발, Hook, Scheduler, 초기화, Minibatch, BatchNorm 개념 정리

강의 핵심 문장:

```text
머신러닝 모델은 레이어 함수 구조와 파라미터 값의 유기적인 결합이다.
```

## 1. 라이브러리 준비

이번 실습에서는 PyTorch의 `nn.Module`, `nn.Linear`, `nn.ReLU`, `nn.Sequential`, `optim.SGD`를 사용합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

%matplotlib inline

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(suppress=True, precision=4)
torch.manual_seed(123)
np.random.seed(123)

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)

코드 설명:

- `torch`: PyTorch 기본 라이브러리
- `torch.nn as nn`: 신경망 레이어와 모델 정의에 사용
- `torch.optim as optim`: Optimizer 사용
- `manual_seed`: 랜덤 결과를 재현 가능하게 고정

## 2. 신경망 개념도 vs PyTorch 프로그램 모델

강의에서는 신경망 그림과 실제 PyTorch 코드 관점을 구분했습니다.

### 신경망 개념도

- 입력층, 은닉층, 출력층의 노드가 연결된 그림
- 사용자가 보기 쉬운 설계도 느낌

### PyTorch 프로그램 모델

- Tensor가 Layer Function을 순서대로 통과하는 코드 구조
- 실제 구현과 가까운 조리 과정 느낌

핵심:

```text
신경망 그림의 Layer와 PyTorch의 Layer Function은 다르게 이해해야 한다.
```

In [ ]:
concept_map = {
    "신경망 개념도": "입력층, 은닉층, 출력층으로 표현되는 설계 관점",
    "PyTorch 프로그램 모델": "Tensor가 함수들을 순서대로 통과하는 구현 관점",
    "Layer Function": "Tensor를 입력받아 Tensor를 출력하는 함수",
    "Parameter": "학습을 통해 값이 조정되는 weight, bias"
}

for key, value in concept_map.items():
    print(f"{key}: {value}")

## 3. 핵심 용어 정리

Day5에서 중요한 용어는 다음 네 가지입니다.

| 용어 | 의미 |
|---|---|
| Layer Function | Tensor를 입력받아 Tensor를 출력하는 함수 |
| Parameter | Layer 내부에서 학습되는 값 |
| Model | Layer Function을 조합한 거대한 합성 함수 |
| Learning | 정답에 가까워지도록 Parameter를 조정하는 과정 |

In [ ]:
terms = [
    ("Layer Function", "nn.Linear, nn.ReLU처럼 Tensor를 변환하는 함수"),
    ("Parameter", "weight와 bias처럼 학습되는 값"),
    ("Model", "Layer Function을 여러 개 연결한 함수"),
    ("Learning", "Loss를 줄이도록 Parameter를 수정하는 과정")
]

for name, desc in terms:
    print(f"{name}: {desc}")

## 4. 손글씨 숫자 분류 모델의 Layer 정의

예시로 28x28 이미지를 숫자 0~9 중 하나로 분류하는 모델을 생각합니다.

이미지 크기:

```text
28 × 28 = 784
```

따라서 입력 크기는 784입니다.

출력은 숫자 0~9의 10개 클래스이므로 10입니다.

In [ ]:
# 첫 번째 선형 함수: 784개 입력을 받아 128개로 출력
l1 = nn.Linear(784, 128)

# 두 번째 선형 함수: 128개 입력을 받아 10개로 출력
l2 = nn.Linear(128, 10)

# 활성화 함수
relu = nn.ReLU(inplace=True)

print(l1)
print(relu)
print(l2)

코드 설명:

- `nn.Linear(784, 128)`: 784차원 입력을 128차원으로 변환합니다.
- `nn.Linear(128, 10)`: 128차원 입력을 10차원 출력으로 변환합니다.
- `nn.ReLU()`: 음수는 0으로 만들고 양수는 그대로 통과시킵니다.

중요:

```text
앞 Layer의 출력 크기와 다음 Layer의 입력 크기가 반드시 맞아야 한다.
```

## 5. 더미 입력 데이터 만들기

100개의 이미지 데이터가 있다고 가정합니다.

각 이미지는 784개의 숫자로 펼쳐진 상태입니다.

shape:

```text
[100, 784]
```

- 100: 데이터 개수 batch size
- 784: 각 데이터의 feature 수

In [ ]:
inputs = torch.randn(100, 784)

print("inputs shape:", inputs.shape)

`torch.randn(100, 784)`는 정규분포 난수로 더미 데이터를 만듭니다.

실제 이미지 데이터는 아니지만, 모델의 입출력 shape을 확인하는 데 충분합니다.

## 6. Layer를 하나씩 통과시키기

입력 Tensor가 Layer를 지나면서 shape이 어떻게 바뀌는지 확인합니다.

In [ ]:
m1 = l1(inputs)
m2 = relu(m1)
outputs = l2(m2)

print("inputs shape:", inputs.shape)
print("m1 shape:", m1.shape)
print("m2 shape:", m2.shape)
print("outputs shape:", outputs.shape)

흐름 해석:

```text
inputs:  [100, 784]
m1:      [100, 128]
m2:      [100, 128]
outputs: [100, 10]
```

- `l1`이 784차원을 128차원으로 바꿉니다.
- `ReLU`는 shape을 바꾸지 않고 값만 바꿉니다.
- `l2`가 128차원을 10차원으로 바꿉니다.

## 7. `nn.Sequential`로 합성 함수 만들기

여러 Layer를 순서대로 연결할 때 `nn.Sequential`을 사용합니다.

이렇게 하면 모델 전체를 하나의 함수처럼 사용할 수 있습니다.

In [ ]:
net2 = nn.Sequential(
    l1,
    relu,
    l2
)

outputs2 = net2(inputs)

print("입력 텐서 shape:", inputs.shape)
print("출력 텐서 shape:", outputs2.shape)

`net2(inputs)`는 내부적으로 다음과 같이 동작합니다.

```text
inputs → l1 → relu → l2 → outputs
```

즉, PyTorch 모델은 여러 Layer Function이 연결된 합성 함수입니다.

## 8. PyTorch 학습의 3대 핵심 함수

강의에서는 PyTorch 학습에 필요한 3대 요소를 강조했습니다.

```text
1. net        : 예측 함수
2. criterion  : 손실 함수
3. optimizer  : 최적화 함수
```

이 세 가지가 없으면 AI 학습은 시작되지 않습니다.

In [ ]:
three_core = {
    "net": "학습 데이터를 입력받아 예측값을 출력하는 모델",
    "criterion": "예측값과 정답을 비교해 손실을 계산하는 함수",
    "optimizer": "gradient를 바탕으로 parameter를 수정하는 도구"
}

for key, value in three_core.items():
    print(f"{key}: {value}")

## 9. 학습 4단계 순환

PyTorch 학습은 다음 4단계를 반복합니다.

```text
1. 예측 계산: outputs = net(inputs)
2. 손실 계산: loss = criterion(outputs, labels)
3. 경사 계산: loss.backward()
4. 파라미터 수정: optimizer.step()
```

In [ ]:
training_loop_steps = [
    "outputs = net(inputs)",
    "loss = criterion(outputs, labels)",
    "loss.backward()",
    "optimizer.step()"
]

for i, step in enumerate(training_loop_steps, 1):
    print(f"{i}. {step}")

실제 학습 루프에서는 여기에 gradient 초기화가 추가됩니다.

```python
optimizer.zero_grad()
```

PyTorch는 gradient를 누적하기 때문에 매 반복마다 초기화해야 합니다.

## 10. 실험 데이터 만들기: y = x² + noise

이번 실습에서는 2차 함수 형태의 데이터를 만듭니다.

목표는 모델이 이 곡선 패턴을 학습할 수 있는지 확인하는 것입니다.

In [ ]:
np.random.seed(123)

x = np.random.randn(100, 1) * 2.5
y = x**2 + np.random.randn(100, 1) * 0.8

x_train = x[:50, :]
y_train = y[:50, :]

x_test = x[50:, :]
y_test = y[50:, :]

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

데이터 설명:

- `x`: 입력값
- `y`: `x²`에 noise를 더한 정답값
- 앞 50개는 훈련 데이터
- 뒤 50개는 검증 데이터

이 데이터는 곡선 패턴을 가지고 있으므로 단순한 직선 모델로는 잘 맞추기 어렵습니다.

## 11. 학습 데이터 산점도 출력

훈련 데이터와 검증 데이터를 그래프로 확인합니다.

In [ ]:
plt.scatter(x_train, y_train, label="train")
plt.scatter(x_test, y_test, marker="x", label="test")
plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.title("Quadratic Data")
plt.show()

그래프 해석:

점들이 직선이 아니라 U자 형태에 가깝게 분포합니다.

따라서 이 데이터는 선형 모델보다 비선형 모델이 더 잘 학습할 수 있습니다.

## 12. NumPy 배열을 Tensor로 변환

PyTorch 모델 학습을 위해 데이터를 Tensor로 변환합니다.

In [ ]:
inputs = torch.tensor(x_train).float()
labels = torch.tensor(y_train).float()

inputs_test = torch.tensor(x_test).float()
labels_test = torch.tensor(y_test).float()

print("inputs shape:", inputs.shape)
print("labels shape:", labels.shape)
print("inputs_test shape:", inputs_test.shape)
print("labels_test shape:", labels_test.shape)

변수 설명:

- `inputs`: 훈련 입력
- `labels`: 훈련 정답
- `inputs_test`: 검증 입력
- `labels_test`: 검증 정답

`float()`를 붙이는 이유는 PyTorch의 `nn.Linear`가 보통 float Tensor를 입력으로 받기 때문입니다.

## 13. 모델 1: 선형 회귀 모델 Net

첫 번째 모델은 가장 단순한 선형 모델입니다.

구조:

```text
Linear(1 → 1)
```

활성화 함수도 없고 은닉층도 없습니다.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(1, 1)

    def forward(self, x):
        x1 = self.l1(x)
        return x1

net = Net()

print(net)

코드 설명:

- `class Net(nn.Module)`: PyTorch 모델 클래스를 정의합니다.
- `super().__init__()`: 부모 클래스인 `nn.Module`을 초기화합니다.
- `self.l1 = nn.Linear(1, 1)`: 입력 1개를 출력 1개로 바꾸는 선형 Layer입니다.
- `forward()`: 입력 데이터가 모델 안에서 어떻게 흐르는지 정의합니다.

## 14. 모델 1 학습 준비

Optimizer와 손실 함수를 정의합니다.

In [ ]:
lr = 0.01

net = Net()
optimizer = optim.SGD(net.parameters(), lr=lr)
criterion = nn.MSELoss()

num_epochs = 1000
history_net = np.zeros((0, 2))

print("optimizer:", optimizer)
print("criterion:", criterion)

코드 설명:

- `net.parameters()`: 모델 내부의 weight와 bias를 가져옵니다.
- `optim.SGD(...)`: 경사하강법 Optimizer입니다.
- `nn.MSELoss()`: 평균 제곱 오차 손실 함수입니다.
- `history_net`: epoch와 loss를 기록합니다.

## 15. 모델 1 학습 루프

PyTorch의 기본 학습 루프입니다.

In [ ]:
for epoch in range(num_epochs):
    optimizer.zero_grad()

    outputs = net(inputs)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        history_net = np.vstack((history_net, np.array([epoch, loss.item()])))
        print(f"Epoch {epoch} loss: {loss.item():.5f}")

학습 루프 설명:

1. `optimizer.zero_grad()`: 이전 gradient 초기화
2. `outputs = net(inputs)`: 예측
3. `loss = criterion(outputs, labels)`: 손실 계산
4. `loss.backward()`: gradient 계산
5. `optimizer.step()`: parameter 업데이트

이 과정이 경사하강법 구현의 핵심입니다.

## 16. 모델 1 결과 그래프

선형 모델이 2차 함수 데이터를 얼마나 맞추는지 확인합니다.

In [ ]:
labels_pred = net(inputs_test)

plt.title("Net: Linear only")
plt.scatter(inputs_test[:, 0].data, labels_pred[:, 0].data, label="prediction")
plt.scatter(inputs_test[:, 0].data, labels_test[:, 0].data, marker="x", label="target")
plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.show()

그래프 해석:

모델 1은 선형 함수 하나만 사용합니다.

따라서 예측값은 직선 형태에 가깝습니다.

하지만 실제 데이터는 U자 곡선이므로 잘 맞추기 어렵습니다.

## 17. 모델 2: 활성화 함수 없는 깊은 모델 Net2

두 번째 모델은 Linear Layer를 3개 사용합니다.

구조:

```text
Linear → Linear → Linear
```

하지만 활성화 함수가 없습니다.

In [ ]:
class Net2(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(1, 10)
        self.l2 = nn.Linear(10, 10)
        self.l3 = nn.Linear(10, 1)

    def forward(self, x):
        x1 = self.l1(x)
        x2 = self.l2(x1)
        x3 = self.l3(x2)
        return x3

net2 = Net2()

print(net2)

중요 개념:

선형 함수만 여러 번 합성해도 결과는 여전히 선형 함수입니다.

```text
Linear + Linear + Linear = Linear
```

따라서 Layer를 많이 쌓아도 활성화 함수가 없으면 복잡한 곡선을 학습하기 어렵습니다.

## 18. 모델 2 학습

활성화 함수 없는 깊은 모델을 학습시킵니다.

In [ ]:
net2 = Net2()
optimizer = optim.SGD(net2.parameters(), lr=lr)
criterion = nn.MSELoss()

history_net2 = np.zeros((0, 2))

for epoch in range(num_epochs):
    optimizer.zero_grad()

    outputs = net2(inputs)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        history_net2 = np.vstack((history_net2, np.array([epoch, loss.item()])))
        print(f"Epoch {epoch} loss: {loss.item():.5f}")

이 모델은 층이 많지만 활성화 함수가 없습니다.

따라서 모델 1보다 구조는 복잡해 보여도 실제 표현력은 크게 달라지지 않습니다.

## 19. 모델 2 결과 그래프

활성화 함수 없는 깊은 모델의 예측 결과를 확인합니다.

In [ ]:
labels_pred2 = net2(inputs_test)

plt.title("Net2: Deep Linear without Activation")
plt.scatter(inputs_test[:, 0].data, labels_pred2[:, 0].data, label="prediction")
plt.scatter(inputs_test[:, 0].data, labels_test[:, 0].data, marker="x", label="target")
plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.show()

그래프 해석:

예측 결과가 여전히 직선에 가깝습니다.

이 실험이 보여주는 핵심은 다음과 같습니다.

```text
활성화 함수가 없으면 깊게 쌓아도 비선형 패턴을 학습하기 어렵다.
```

## 20. 모델 3: ReLU 활성화 함수가 있는 모델 Net3

세 번째 모델은 Linear Layer 사이에 ReLU를 넣습니다.

구조:

```text
Linear → ReLU → Linear → ReLU → Linear
```

이제 모델은 비선형 패턴을 학습할 수 있습니다.

In [ ]:
class Net3(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(1, 10)
        self.l2 = nn.Linear(10, 10)
        self.l3 = nn.Linear(10, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x1 = self.relu(self.l1(x))
        x2 = self.relu(self.l2(x1))
        x3 = self.l3(x2)
        return x3

net3 = Net3()

print(net3)

코드 설명:

- `self.relu = nn.ReLU(inplace=True)`: ReLU 활성화 함수입니다.
- `self.relu(self.l1(x))`: Linear 결과에 ReLU를 적용합니다.
- 마지막 Layer 뒤에는 회귀 출력값을 그대로 사용하기 위해 ReLU를 붙이지 않습니다.

## 21. 모델 3 학습

활성화 함수가 있는 모델을 학습시킵니다.

In [ ]:
net3 = Net3()
optimizer = optim.SGD(net3.parameters(), lr=lr)
criterion = nn.MSELoss()

history_net3 = np.zeros((0, 2))

for epoch in range(num_epochs):
    optimizer.zero_grad()

    outputs = net3(inputs)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        history_net3 = np.vstack((history_net3, np.array([epoch, loss.item()])))
        print(f"Epoch {epoch} loss: {loss.item():.5f}")

모델 3은 활성화 함수가 있기 때문에 곡선 형태의 데이터를 더 잘 표현할 수 있습니다.

이 실습의 핵심은 모델 깊이보다 **비선형성**이 중요하다는 점입니다.

## 22. 모델 3 결과 그래프

ReLU가 있는 모델의 예측 결과를 확인합니다.

In [ ]:
labels_pred3 = net3(inputs_test)

plt.title("Net3: Deep Network with ReLU")
plt.scatter(inputs_test[:, 0].data, labels_pred3[:, 0].data, label="prediction")
plt.scatter(inputs_test[:, 0].data, labels_test[:, 0].data, marker="x", label="target")
plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.show()

그래프 해석:

모델 3은 U자 형태의 곡선 패턴을 훨씬 더 잘 따라갑니다.

즉, ReLU 활성화 함수가 모델에 비선형성을 추가했기 때문입니다.

## 23. 세 모델의 학습 곡선 비교

세 모델의 Loss 변화를 한 번에 비교합니다.

In [ ]:
plt.plot(history_net[:, 0], history_net[:, 1], label="Net")
plt.plot(history_net2[:, 0], history_net2[:, 1], label="Net2")
plt.plot(history_net3[:, 0], history_net3[:, 1], label="Net3")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training Loss Comparison")
plt.legend()
plt.show()

그래프 해석:

- `Net`: 선형 모델이라 표현력이 낮습니다.
- `Net2`: 깊지만 활성화 함수가 없어 여전히 선형에 가깝습니다.
- `Net3`: ReLU 덕분에 비선형 패턴을 학습할 수 있습니다.

결론:

```text
딥러닝에서 활성화 함수는 선택이 아니라 핵심 구조이다.
```

## 24. 순전파 Forward Pass

순전파는 입력 데이터가 모델의 각 Layer를 통과하여 최종 출력을 만드는 과정입니다.

```text
Input → Layer → Activation → Output
```

PyTorch에서는 `outputs = net(inputs)`가 순전파입니다.

In [ ]:
sample_input = torch.tensor([[1.0]])

sample_output = net3(sample_input)

print("sample_input:", sample_input)
print("sample_output:", sample_output)

코드 설명:

- `sample_input`: 입력 Tensor
- `net3(sample_input)`: 모델을 통과한 예측값
- 이 호출이 내부적으로 `forward()`를 실행합니다.

즉, PyTorch 모델은 객체처럼 보이지만 함수처럼 호출할 수 있습니다.

## 25. 역전파 Backward Pass

역전파는 최종 손실에서 시작해 거꾸로 올라가며 각 파라미터의 책임을 계산하는 과정입니다.

```text
Loss → Output Layer → Hidden Layer → Parameters
```

PyTorch에서는 `loss.backward()`가 역전파입니다.

In [ ]:
# 작은 예시로 gradient 확인
net3.zero_grad()

sample_pred = net3(inputs[:5])
sample_loss = criterion(sample_pred, labels[:5])
sample_loss.backward()

for name, param in net3.named_parameters():
    print(name, "grad mean:", param.grad.mean().item())

코드 설명:

- `net3.zero_grad()`: 기존 gradient 초기화
- `sample_loss.backward()`: 역전파 실행
- `named_parameters()`: 모델 안의 파라미터 이름과 값을 확인
- `param.grad`: 각 파라미터의 gradient

이 gradient를 바탕으로 Optimizer가 파라미터를 수정합니다.

## 26. Autograd Hook으로 gradient 관찰하기

Hook은 역전파 중 gradient를 중간에서 확인할 수 있는 기능입니다.

강의에서는 CCTV처럼 각 층의 gradient 흐름을 관찰하는 도구로 설명했습니다.

In [ ]:
def gradient_hook(grad):
    print("gradient mean:", grad.mean().item())
    print("gradient std:", grad.std().item())
    return grad

# l1 weight에 hook 등록
hook_handle = net3.l1.weight.register_hook(gradient_hook)

net3.zero_grad()
sample_pred = net3(inputs[:10])
sample_loss = criterion(sample_pred, labels[:10])
sample_loss.backward()

# hook 제거
hook_handle.remove()

Hook 설명:

- `register_hook()`: 특정 Tensor의 gradient가 계산될 때 실행할 함수를 등록합니다.
- `grad.mean()`: gradient 평균
- `grad.std()`: gradient 표준편차

이를 통해 기울기 소실이나 폭발 여부를 관찰할 수 있습니다.

## 27. 학습률 Scheduler 예시

학습률은 한 번에 얼마나 이동할지 정하는 값입니다.

너무 크면 발산하고, 너무 작으면 학습이 느립니다.

Scheduler는 학습 중 learning rate를 조절합니다.

In [ ]:
scheduler_net = Net3()
scheduler_optimizer = optim.SGD(scheduler_net.parameters(), lr=0.1)

scheduler = optim.lr_scheduler.StepLR(
    scheduler_optimizer,
    step_size=10,
    gamma=0.1
)

for epoch in range(25):
    current_lr = scheduler_optimizer.param_groups[0]["lr"]
    if epoch in [0, 9, 10, 19, 20, 24]:
        print(f"epoch {epoch}, lr = {current_lr}")

    scheduler.step()

코드 설명:

- `StepLR`: 일정 epoch마다 learning rate를 줄입니다.
- `step_size=10`: 10 epoch마다 감소
- `gamma=0.1`: learning rate를 0.1배로 감소

전략:

```text
초반에는 크게 이동
후반에는 작게 이동
```

## 28. 가중치 초기화 예시

가중치 초기화는 학습 시작점을 정하는 과정입니다.

잘못 초기화하면 학습이 잘 안 될 수 있습니다.

대표 초기화:

- Xavier: Tanh, Sigmoid 계열에 적합
- He/Kaiming: ReLU 계열에 적합

In [ ]:
init_layer = nn.Linear(10, 5)

# ReLU 계열에 자주 사용하는 He/Kaiming 초기화
nn.init.kaiming_normal_(init_layer.weight, mode="fan_in", nonlinearity="relu")

print("weight mean:", init_layer.weight.mean().item())
print("weight std:", init_layer.weight.std().item())

코드 설명:

- `nn.init.kaiming_normal_`: He 초기화
- ReLU를 사용하는 네트워크에서 자주 사용합니다.
- `_`가 붙은 함수는 Tensor를 직접 수정하는 in-place 함수입니다.

## 29. Minibatch 개념

전체 데이터를 한 번에 학습하는 대신 작은 묶음으로 나누어 학습하는 방식입니다.

예:

```text
10,000개 데이터
batch size = 100
→ 1 epoch 동안 100번 업데이트
```

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(inputs, labels)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

for batch_x, batch_y in loader:
    print("batch_x shape:", batch_x.shape)
    print("batch_y shape:", batch_y.shape)
    break

코드 설명:

- `TensorDataset`: 입력과 정답 Tensor를 묶습니다.
- `DataLoader`: minibatch 단위로 데이터를 꺼냅니다.
- `batch_size=16`: 한 번에 16개씩 학습합니다.
- `shuffle=True`: 매 epoch마다 순서를 섞습니다.

Minibatch는 메모리 효율성과 학습 속도를 개선합니다.

## 30. Batch Normalization 예시

BatchNorm은 각 층의 입력 분포를 안정화하여 학습을 더 빠르고 안정적으로 만들 수 있습니다.

일반적인 순서:

```text
Linear → BatchNorm → ReLU
```

In [ ]:
bn_model = nn.Sequential(
    nn.Linear(1, 10),
    nn.BatchNorm1d(10),
    nn.ReLU(),
    nn.Linear(10, 1)
)

sample_batch = torch.randn(16, 1)
sample_output = bn_model(sample_batch)

print(bn_model)
print("sample_output shape:", sample_output.shape)

BatchNorm 효과:

- gradient flow 개선
- 학습률을 더 크게 사용할 수 있음
- 초기화 민감도 감소
- 약한 정규화 효과

주의:

RNN이나 Transformer에서는 BatchNorm보다 LayerNorm이 더 자주 사용됩니다.

## 31. 주요 함수 / 변수 / 약어 정리

| 이름 | 의미 | 설명 |
|---|---|---|
| `nn` | Neural Network | PyTorch 신경망 모듈 |
| `nn.Module` | Model Base Class | PyTorch 모델의 부모 클래스 |
| `nn.Linear` | Linear Layer | 선형 변환 Layer |
| `nn.ReLU` | ReLU Activation | 비선형 활성화 함수 |
| `nn.Sequential` | Sequential Model | Layer를 순서대로 묶는 함수 |
| `net` | Prediction Function | 예측 함수, 모델 |
| `criterion` | Loss Function | 손실 함수 |
| `optimizer` | Optimizer | 파라미터 수정 도구 |
| `SGD` | Stochastic Gradient Descent | 경사하강법 계열 Optimizer |
| `MSELoss` | Mean Squared Error Loss | 평균 제곱 오차 |
| `inputs` | Input Tensor | 입력 데이터 |
| `labels` | Target Tensor | 정답 데이터 |
| `outputs` | Output Tensor | 예측값 |
| `loss` | Loss | 오차 |
| `forward` | Forward Pass | 순전파 |
| `backward` | Backward Pass | 역전파 |
| `grad` | Gradient | 기울기 |
| `lr` | Learning Rate | 학습률 |
| `epoch` | Epoch | 전체 학습 반복 단위 |
| `BatchNorm` | Batch Normalization | 배치 정규화 |
| `LayerNorm` | Layer Normalization | 샘플별 특징 차원 정규화 |

## 32. 시험용 요약

```text
PyTorch 학습의 3대 요소 = net + criterion + optimizer
```

핵심 정리:

- Layer Function은 Tensor를 입력받아 Tensor를 출력하는 함수입니다.
- Parameter는 Layer 내부에서 학습되는 weight와 bias입니다.
- Model은 Layer Function들을 연결한 합성 함수입니다.
- `nn.Linear(in, out)`은 입력 차원을 출력 차원으로 바꾸는 선형 Layer입니다.
- Layer의 출력 크기와 다음 Layer의 입력 크기는 반드시 맞아야 합니다.
- `nn.Sequential`은 여러 Layer를 순서대로 묶습니다.
- 학습 4단계는 예측 계산 → 손실 계산 → 경사 계산 → 파라미터 수정입니다.
- `optimizer.zero_grad()`는 gradient 초기화입니다.
- `outputs = net(inputs)`는 순전파입니다.
- `loss.backward()`는 역전파입니다.
- `optimizer.step()`은 파라미터 업데이트입니다.
- 선형 함수만 여러 번 합성해도 결국 선형 함수입니다.
- 활성화 함수는 모델에 비선형성을 부여합니다.
- ReLU가 있는 모델은 2차 함수 같은 곡선 패턴을 더 잘 학습할 수 있습니다.
- Hook은 gradient 흐름을 관찰하는 도구입니다.
- Scheduler는 학습률을 조절합니다.
- He 초기화는 ReLU 계열에 자주 사용합니다.
- Minibatch는 데이터를 작은 묶음으로 나누어 학습하는 방식입니다.
- BatchNorm은 학습 안정성과 gradient flow 개선에 도움을 줍니다.